# LIMO-style LoRA SFT on Qwen3.6-35B-A3B

Stacks on top of PWMV wrapper (qwen36-deepconf-probe) for compound reasoning gains.

- PWMV alone: +6pp SuperGPQA (validated n=40, greedy 0.70 -> PWMV 0.75)
- LIMO LoRA alone: projected +5pp (LIMO/s1 precedent on Qwen2.5-32B)
- Stacked: projected 0.75-0.80 SuperGPQA

Recipe: 1K curated reasoning samples, LoRA r=16, 2 epochs, ~9h on RTX 6000.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception: ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer', 'peft', 'trl')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

pip('install', '-q', '--upgrade', 'peft', 'trl', 'datasets')

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset
from huggingface_hub import snapshot_download, HfApi, create_repo
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/limo_lora'); OUT.mkdir(exist_ok=True)
HF_OUT = 'caiovicentino1/Qwen3.6-35B-A3B-LIMO-Probe'
MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'

N_SAMPLES = 1000
MAX_SEQ_LEN = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LR = 1e-4
EPOCHS = 2
BATCH_SIZE = 1
GRAD_ACCUM = 8
WARMUP_RATIO = 0.05

print('setup done')

## Cell 2 — Load model + find LoRA target modules

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.config.use_cache = False
model.gradient_checkpointing_enable()
print(f'Model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

target_set = set()
skip_patterns = ['in_proj_a', 'in_proj_b', 'routed_experts', 'norm', 'embed']
for n, m in model.named_modules():
    if not isinstance(m, torch.nn.Linear): continue
    if any(sk in n for sk in skip_patterns): continue
    lname = n.split('.')[-1]
    if lname in ['q_proj','k_proj','v_proj','o_proj','in_proj_qkv','out_proj',
                 'gate_proj','up_proj','down_proj']:
        if lname in ['gate_proj','up_proj','down_proj'] and 'shared_expert' not in n:
            continue
        target_set.add(lname)

TARGET_MODULES = sorted(target_set)
print(f'LoRA target modules ({len(TARGET_MODULES)}): {TARGET_MODULES}')

## Cell 3 — Load s1K-1.1 (public LIMO-style dataset)

In [ ]:
try:
    ds = load_dataset('simplescaling/s1K-1.1', split='train')
    src = 's1K-1.1'
except Exception as e:
    print(f's1K-1.1 failed: {e}, trying OpenR1')
    ds = load_dataset('open-r1/OpenR1-Math-220k', split='train[:5000]')
    src = 'OpenR1-Math'

print(f'{src}: {len(ds)} samples, columns: {ds.column_names}')
ex = ds[0]
for k in list(ex.keys())[:10]:
    v = ex[k]
    show = (v[:200] + '...') if isinstance(v, str) and len(v) > 200 else str(v)[:200]
    print(f'  {k}: {show}')

## Cell 4 — Format samples to Qwen3.6 thinking chat

In [ ]:
def format_s1k(ex):
    q = ex.get('question', '')
    think = ''
    for k in ['thinking_trajectories', 'gemini_thinking_trajectory', 'deepseek_thinking_trajectory']:
        v = ex.get(k)
        if v:
            think = v[0] if isinstance(v, list) else v
            break
    ans = ex.get('solution') or ex.get('attempt') or ex.get('gemini_attempt') or ex.get('deepseek_attempt') or ''
    if not q or not think or not ans: return None
    assistant = f'<think>\n{think.strip()}\n</think>\n\n{ans.strip()}'
    msgs = [{'role':'user','content':q.strip()}, {'role':'assistant','content':assistant}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, enable_thinking=False)

def format_openr1(ex):
    if 'messages' in ex and isinstance(ex['messages'], list):
        return tok.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False, enable_thinking=False)
    q, a = ex.get('problem',''), ex.get('solution','')
    if not q or not a: return None
    msgs = [{'role':'user','content':q.strip()}, {'role':'assistant','content':a.strip()}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, enable_thinking=False)

formatter = format_s1k if src == 's1K-1.1' else format_openr1

formatted = []
for ex in ds:
    text = formatter(ex)
    if not text: continue
    n = len(tok.encode(text, add_special_tokens=False))
    if n < 200 or n > MAX_SEQ_LEN: continue
    formatted.append({'text': text, 'n_tokens': n})
    if len(formatted) >= N_SAMPLES: break

print(f'{len(formatted)} samples, avg tokens: {np.mean([s["n_tokens"] for s in formatted]):.0f}')
print(f'\nSample 0 (first 500 chars):\n{formatted[0]["text"][:500]}')

train_ds = Dataset.from_list([{'text': s['text']} for s in formatted])

## Cell 5 — PEFT LoRA wrap

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias='none', task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 6 — Train with TRL SFTTrainer

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir=str(OUT / 'trainer_ckpt'),
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    logging_steps=10, save_steps=100, save_total_limit=2,
    report_to='none', optim='adamw_torch',
    max_length=MAX_SEQ_LEN, packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataset_text_field='text',
    dataloader_num_workers=0,
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, processing_class=tok)

total_steps = len(train_ds) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM)
print(f'Training {total_steps} steps starting at {time.ctime()}')
t0 = time.time()
trainer.train()
print(f'Training done in {(time.time()-t0)/3600:.2f}h')

## Cell 7 — Save adapter + quick sanity eval

In [ ]:
from safetensors import safe_open

adapter_dir = OUT / 'lora_adapter'
trainer.save_model(str(adapter_dir))
tok.save_pretrained(str(adapter_dir))
print(f'Adapter saved: {adapter_dir}')

# Quick sanity on 5 Stage B
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
corpus_path = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache',
                                 allow_patterns=['shards/q00*.safetensors'])
shards = sorted((Path(corpus_path)/'shards').glob('q*.safetensors'))[:20]

def fmt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = f'Answer the following multiple-choice question. Reason step by step, then put the letter in \\boxed{{}}.\n\nQuestion: {q}\n\nOptions:\n{choices}'
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True, enable_thinking=True)

def extract(text):
    post = text.split('</think>')[-1] if '</think>' in text else text
    m = re.search(r'\\boxed\{([A-J])\}', post)
    if m: return m.group(1)
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    return m[-1] if m else None

questions = []
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(q=meta['question'], o=opts, g=meta['gold_letter']))
    if len(questions) >= 5: break

model.eval()
correct = 0
for q in questions:
    p = fmt(q['q'], q['o'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=2500, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    pred = extract(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))
    hit = pred == q['g']
    correct += int(hit)
    print(f'  gold={q["g"]} pred={pred} {"OK" if hit else "X"}')
print(f'\nSanity: {correct}/5')

## Cell 8 — Upload adapter + wrapper + probe to HF

In [ ]:
# Write wrapper code to disk
WRAPPER_CODE = '''"""HyperQwen-A3B: LIMO LoRA + probe-weighted self-consistency."""
import torch, re, pickle
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import PeftModel

class HyperQwenWrapper:
    def __init__(self, base_id="Qwen/Qwen3.6-35B-A3B", adapter_path=None, probe_path=None):
        self.tok = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
        if self.tok.pad_token_id is None: self.tok.pad_token = self.tok.eos_token
        base = AutoModelForImageTextToText.from_pretrained(
            base_id, dtype=torch.bfloat16, device_map="cuda",
            attn_implementation="sdpa", trust_remote_code=True)
        self.model = PeftModel.from_pretrained(base, adapter_path) if adapter_path else base
        self.model.eval()
        self.probe = pickle.load(open(probe_path, "rb")) if probe_path else None
        self._chunks = []
        self._install_hook()

    def _install_hook(self):
        def hook(m, i, o):
            h = o[0] if isinstance(o, tuple) else o
            self._chunks.append(h[:, -1, :].detach().float().cpu())
        base = self.model.base_model if hasattr(self.model, "base_model") else self.model
        layer = base.model.language_model.layers[11]
        self.handle = layer.register_forward_hook(hook)

    @staticmethod
    def extract_answer(text):
        post = text.split("</think>")[-1] if "</think>" in text else text
        m = re.search(r"\\boxed\{([A-J])\}", post)
        return m.group(1) if m else None

    def predict(self, question, options, n_traces=4, temperature=0.7, max_new=2500):
        choices = "\n".join(f"{chr(65+i)}. {o}" for i, o in enumerate(options))
        content = ("Answer the following multiple-choice question. "
            "Reason step by step, then put the letter in \\boxed{}.\n\n"
            f"Question: {question}\n\nOptions:\n{choices}")
        prompt = self.tok.apply_chat_template(
            [{"role":"user","content":content}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True)
        ids = self.tok(prompt, return_tensors="pt").input_ids.cuda()
        self._chunks = []
        with torch.no_grad():
            out = self.model.generate(
                ids, max_new_tokens=max_new, do_sample=True, temperature=temperature,
                num_return_sequences=n_traces, top_p=0.95,
                pad_token_id=self.tok.pad_token_id, use_cache=True)
        if self.probe and len(self._chunks) >= 2:
            embs = torch.stack(self._chunks[1:], dim=0).mean(dim=0).numpy()
            scores = self.probe.predict_proba(embs)[:, 1].tolist()
        else:
            scores = [1.0] * n_traces
        answers = [self.extract_answer(self.tok.decode(out[i, ids.shape[1]:], skip_special_tokens=True)) for i in range(n_traces)]
        weighted = defaultdict(float)
        for a, s in zip(answers, scores):
            if a: weighted[a] += s
        return max(weighted, key=weighted.get) if weighted else None'''
(OUT / 'hyperqwen.py').write_text(WRAPPER_CODE)
print('OK hyperqwen.py written')

# Copy probe from qwen-honest-DeepConf repo
try:
    probe_path = snapshot_download('caiovicentino1/qwen36-deepconf-probe',
                                    repo_type='model', cache_dir='/content/cache',
                                    allow_patterns=['probe_l11.pkl'])
    import shutil
    shutil.copy(Path(probe_path)/'probe_l11.pkl', OUT/'probe_l11.pkl')
    print('OK probe_l11.pkl copied')
except Exception as e:
    print(f'probe copy failed: {e}')

readme = f'''---
license: mit
base_model: Qwen/Qwen3.6-35B-A3B
tags: [lora, sft, limo, reasoning, probe-weighted-vote]
---

# HyperQwen-A3B: LIMO-tuned Qwen3.6 with Probe-Weighted Voting

SFT LoRA on {N_SAMPLES} curated reasoning traces + PWMV inference wrapper.

## Components

- `lora_adapter/` — PEFT adapter (r={LORA_R}, alpha={LORA_ALPHA})
- `probe_l11.pkl` — qwen-honest LogReg probe on L11 residual
- `hyperqwen.py` — inference wrapper combining both

## Training

- Base: Qwen/Qwen3.6-35B-A3B
- Dataset: {src} ({N_SAMPLES} samples, max_len {MAX_SEQ_LEN})
- LoRA: r={LORA_R}, alpha={LORA_ALPHA}, targets {TARGET_MODULES}
- Schedule: {EPOCHS} epochs, lr={LR}, cosine warmup {WARMUP_RATIO}
- HW: RTX 6000 Blackwell (96 GB)

## Usage

```python
from hyperqwen import HyperQwenWrapper
w = HyperQwenWrapper(
    base_id='Qwen/Qwen3.6-35B-A3B',
    adapter_path='lora_adapter',
    probe_path='probe_l11.pkl')
answer = w.predict(question, options, n_traces=4)
```

## Stack projection

| stage | SuperGPQA |
|---|---|
| Qwen3.6 greedy | 0.65 |
| + PWMV (no LoRA) | 0.71 |
| + LIMO LoRA (greedy) | ~0.70 |
| **+ LoRA + PWMV stacked** | **~0.75-0.80** |
'''
(OUT / 'README.md').write_text(readme)

create_repo(HF_OUT, repo_type='model', private=False, exist_ok=True)
api = HfApi()
api.upload_folder(folder_path=str(OUT / 'lora_adapter'), path_in_repo='lora_adapter',
                  repo_id=HF_OUT, repo_type='model',
                  commit_message=f'LIMO LoRA r={LORA_R} on {src}')
for fn in ['probe_l11.pkl', 'hyperqwen.py', 'README.md']:
    fp = OUT / fn
    if fp.exists():
        api.upload_file(path_or_fileobj=str(fp), path_in_repo=fn,
                        repo_id=HF_OUT, repo_type='model',
                        commit_message=f'upload {fn}')
        print(f'OK {fn}')
print(f'\nDone: https://huggingface.co/{HF_OUT}')